In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

# Load final model-ready dataset
df = pd.read_csv("top_3_data/top3_model_ready_training_data.csv")

df["visit_date"] = pd.to_datetime(df["visit_date"])

df.head()

,air_store_id,visit_date,visitors,day_of_week,holiday_flg,month,day,year,week_of_year,day_of_week_num,...,reserved_visitors_mean,avg_reserve_lead_days,has_reservation_data,visitors_lag_1,visitors_lag_7,visitors_lag_14,visitors_lag_28,visitors_rolling_7_mean,visitors_rolling_14_mean,visitors_rolling_28_mean
0,air_36bcf77d3382d36e,2016-01-30,45,Saturday,0,1,30,2016,4,5,...,0.0,0.0,0,31.0,51.0,50.0,34.0,30.714286,28.357143,28.928571
1,air_36bcf77d3382d36e,2016-01-31,56,Sunday,0,1,31,2016,4,6,...,0.0,0.0,0,45.0,47.0,49.0,43.0,29.857143,28.000000,29.321429
2,air_36bcf77d3382d36e,2016-02-01,17,Monday,0,2,1,2016,5,0,...,0.0,0.0,0,56.0,30.0,11.0,30.0,31.142857,28.500000,29.785714
3,air_36bcf77d3382d36e,2016-02-02,9,Tuesday,0,2,2,2016,5,1,...,0.0,0.0,0,17.0,14.0,20.0,38.0,29.285714,28.928571,29.321429
4,air_36bcf77d3382d36e,2016-02-03,16,Wednesday,0,2,3,2016,5,2,...,0.0,0.0,0,9.0,20.0,17.0,22.0,28.571429,28.142857,28.285714


### time-based train-test split

In [3]:
# Sort by date
df = df.sort_values("visit_date")

# Use last 39 days as test period
test_start_date = df["visit_date"].max() - pd.Timedelta(days=38)

train_df = df[df["visit_date"] < test_start_date]
test_df = df[df["visit_date"] >= test_start_date]

print("Training rows:", train_df.shape)
print("Testing rows:", test_df.shape)
print("Test start date:", test_start_date)

Training rows: (1229, 27)
Testing rows: (116, 27)
Test start date: 2017-03-15 00:00:00


In [4]:
target = "visitors"

features = [
    "holiday_flg",
    "month",
    "day",
    "week_of_year",
    "day_of_week_num",
    "is_weekend",
    "latitude",
    "longitude",
    "reserved_visitors_sum",
    "reservation_count",
    "reserved_visitors_mean",
    "avg_reserve_lead_days",
    "has_reservation_data",
    "visitors_lag_1",
    "visitors_lag_7",
    "visitors_lag_14",
    "visitors_lag_28",
    "visitors_rolling_7_mean",
    "visitors_rolling_14_mean",
    "visitors_rolling_28_mean",
    "air_store_id",
    "air_genre_name",
    "air_area_name"
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [5]:
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)

# Make sure both train and test have same columns
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

In [6]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=12,
    min_samples_leaf=3
)

model.fit(X_train_encoded, y_train)

predictions = model.predict(X_test_encoded)

# Prevent negative predictions
predictions = np.maximum(predictions, 0)

In [7]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
rmsle = np.sqrt(mean_squared_log_error(y_test, predictions))

print("MAE:", mae)
print("RMSE:", rmse)
print("RMSLE:", rmsle)

MAE: 8.539585499140273
RMSE: 11.434243546531677
RMSLE: 0.4089646069483543


In [8]:
results = test_df[["air_store_id", "visit_date", "visitors"]].copy()
results["predicted_visitors"] = predictions.round(0).astype(int)
results["error"] = abs(results["visitors"] - results["predicted_visitors"])

results.head(20)

,air_store_id,visit_date,visitors,predicted_visitors,error
409,air_36bcf77d3382d36e,2017-03-15,21,19,2
1307,air_a083834e7ffe187e,2017-03-15,18,15,3
858,air_5c817ef28f236bdf,2017-03-15,12,31,19
410,air_36bcf77d3382d36e,2017-03-16,24,20,4
1308,air_a083834e7ffe187e,2017-03-16,8,14,6
859,air_5c817ef28f236bdf,2017-03-16,30,30,0
411,air_36bcf77d3382d36e,2017-03-17,27,27,0
860,air_5c817ef28f236bdf,2017-03-17,54,49,5
1309,air_a083834e7ffe187e,2017-03-17,43,46,3
861,air_5c817ef28f236bdf,2017-03-18,71,82,11


In [9]:
daily_predictions = results.copy()

daily_predictions["week_start"] = (
    daily_predictions["visit_date"] 
    - pd.to_timedelta(daily_predictions["visit_date"].dt.weekday, unit="D")
)

daily_predictions["week_end"] = daily_predictions["week_start"] + pd.Timedelta(days=6)

weekly_predictions = (
    daily_predictions
    .groupby(["air_store_id", "week_start", "week_end"])
    .agg(
        actual_weekly_visitors=("visitors", "sum"),
        predicted_weekly_visitors=("predicted_visitors", "sum"),
        average_daily_error=("error", "mean")
    )
    .reset_index()
)

weekly_predictions

,air_store_id,week_start,week_end,actual_weekly_visitors,predicted_weekly_visitors,average_daily_error
0,air_36bcf77d3382d36e,2017-03-13,2017-03-19,143,185,10.800000
1,air_36bcf77d3382d36e,2017-03-20,2017-03-26,229,234,8.428571
2,air_36bcf77d3382d36e,2017-03-27,2017-04-02,192,217,7.000000
3,air_36bcf77d3382d36e,2017-04-03,2017-04-09,230,197,6.428571
4,air_36bcf77d3382d36e,2017-04-10,2017-04-16,231,202,7.857143
5,air_36bcf77d3382d36e,2017-04-17,2017-04-23,157,175,8.666667
6,air_5c817ef28f236bdf,2017-03-13,2017-03-19,255,242,14.600000
7,air_5c817ef28f236bdf,2017-03-20,2017-03-26,299,314,13.857143
8,air_5c817ef28f236bdf,2017-03-27,2017-04-02,345,294,12.714286
9,air_5c817ef28f236bdf,2017-04-03,2017-04-09,299,283,8.000000


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
import numpy as np
import pandas as pd

# Make sure predictions are not negative
predictions = np.maximum(predictions, 0)

# Evaluation metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
rmsle = np.sqrt(mean_squared_log_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

# Practical accuracy estimate
mean_actual_visitors = y_test.mean()
practical_accuracy = (1 - (mae / mean_actual_visitors)) * 100

print("MODEL EVALUATION RESULTS")
print("-------------------------")
print(f"MAE  - Average error in visitors: {mae:.2f}")
print(f"RMSE - Root mean squared error: {rmse:.2f}")
print(f"RMSLE - Log error score: {rmsle:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"Mean actual visitors: {mean_actual_visitors:.2f}")
print(f"Practical accuracy estimate: {practical_accuracy:.2f}%")

MODEL EVALUATION RESULTS
-------------------------
MAE  - Average error in visitors: 8.54
RMSE - Root mean squared error: 11.43
RMSLE - Log error score: 0.4090
R² Score: 0.6866
Mean actual visitors: 33.28
Practical accuracy estimate: 74.34%
